# Probabilistic Diagnosis Demo (Clean, Full KB v6.5)

This notebook is the **clean working demo** for your workflow.

## Key behaviors implemented
- **Direct vs upstream causes**:
  - Direct causes = `candidate_root_cause_ids` listed under the error code
  - Upstream causes are optionally added (dynamic proposal), but only when context suggests so

- **Warnings**:
  - If a warning is in the initial input, it is treated as trusted evidence and will **not** be asked again
  - If missing, a warning-check procedure may be suggested (when useful)

- **Focused procedure selection**:
  - The model focuses checks/tests on the **highest-ranked direct causes**
  - It prefers procedures whose `targets_root_cause_ids` overlap those causes

- **Evidence updates**:
  - Check/test outcomes update the posterior using Bayes
  - Targeted negative outcomes apply a **near-zero penalty** (multiply probability by 0.01) instead of hard zero

- **Repairs**:
  - Default repair outcomes are standardized:
    - `error_resolved` / `error_not_resolved` (OBS_RESOLVED)
  - Pre-check actions (e.g., remove/reinstall printhead):
    - outcomes `done` / `not_done`
    - if `not_done`, we downweight the currently focused hypothesis and continue

- **Pre-visit parts**:
  - Shows **top 5 parts**
  - Alternatives are shown only if at least **2 of top-5 causes** have p ≥ 0.2


In [2]:
import json, math, uuid, datetime, re
from pathlib import Path

KB_PATH = Path(r"C:/Project/CPP_CORE/kb/kb_ink_system_full_v6_5.json")
kb = json.loads(KB_PATH.read_text(encoding="utf-8"))

root_causes = {rc["root_cause_id"]: rc for rc in kb["catalogs"]["root_causes"]}
obs_catalog  = {o["observation_id"]: o for o in kb["catalogs"]["observations"]}
procedures   = {p["procedure_id"]: p for p in kb["catalogs"]["procedures"]}
error_models = {m["error_code"]: m for m in kb["catalogs"]["error_code_models"]}
parts        = {p["part_id"]: p for p in kb["catalogs"].get("parts", [])}

likelihoods    = kb["parameters"].get("likelihoods", {})
action_success = kb["parameters"].get("action_success", {})

cfg = kb.get("inference_config", {})
EPSILON = cfg.get("stop_condition", {}).get("epsilon_default", 0.02)
REPAIR_RECOMMEND = cfg.get("repair_recommendation", {}).get("recommend_threshold", 0.80)
REPAIR_CONSIDER  = cfg.get("repair_recommendation", {}).get("consider_threshold", 0.75)

UP_CFG   = cfg.get("upstream_generation", {})
EXP_CFG  = cfg.get("expensive_repair_policy", {})
CONFIRM_MAP = EXP_CFG.get("confirming_procedures_by_repair", {})
CONFIRM_FALLBACK = EXP_CFG.get("fallback_if_no_mapping", "any_check_or_test")

WARN_CFG = cfg.get("warning_policy", {})
WARNING_OBS_IDS = set(WARN_CFG.get("warning_observation_ids", ["OBS_WARNING_290005_SEEN","OBS_WARNING_290020_SEEN"]))
WARNING_CHECK_FLAG = WARN_CFG.get("warning_check_identifier", "is_warning_check")

PV_CFG = cfg.get("previsit_parts", {})
PV_TOPK = PV_CFG.get("top_k", 5)
ALT = PV_CFG.get("alternatives_rule", {})
ALT_TOPN = ALT.get("top_n", 5)
ALT_TH   = ALT.get("threshold", 0.20)
ALT_MIN  = ALT.get("min_causes_required", 2)

NEAR_ZERO = 0.01  # near-zero penalty factor for negative targeted checks

print("Loaded KB:", kb["meta"]["name"])
print("Error codes:", len(error_models), "| Causes:", len(root_causes), "| Procedures:", len(procedures), "| Parts:", len(parts))
print("epsilon:", EPSILON, "| recommend:", REPAIR_RECOMMEND, "| consider:", REPAIR_CONSIDER)


Loaded KB: Ink system KB full (v6.5, targeting + resolved outcomes + warning check flag)
Error codes: 41 | Causes: 54 | Procedures: 115 | Parts: 55
epsilon: 0.02 | recommend: 0.8 | consider: 0.75


In [3]:
def now_utc_iso():
    return datetime.datetime.now(datetime.UTC).isoformat().replace("+00:00", "Z")

def normalize(dist: dict) -> dict:
    s = sum(dist.values())
    if s <= 0:
        n = len(dist)
        return {k: 1.0/n for k in dist} if n else {}
    return {k: v/s for k, v in dist.items()}

def entropy(dist: dict) -> float:
    h = 0.0
    for p in dist.values():
        if p > 1e-12:
            h -= p * math.log(p, 2)
    return h

def top_k(dist: dict, k: int = 10):
    return sorted(dist.items(), key=lambda kv: kv[1], reverse=True)[:k]

def direct_cause_ids(error_code: str) -> list[str]:
    return list(error_models[error_code].get("candidate_root_cause_ids", []))

def is_upstream(cause_id: str, error_code: str) -> bool:
    return cause_id not in set(direct_cause_ids(error_code))


In [4]:
# Build lightweight observation models from targeted procedures.
# If the KB lacks explicit likelihood tables, we use these as defaults to make IG meaningful.

OBS_MODELS = {}
for pid, proc in procedures.items():
    if proc.get("kind") not in ("check","test"):
        continue
    obs_id = proc.get("produces_observation")
    targets = proc.get("targets_root_cause_ids") or []
    supp = proc.get("supporting_outcomes") or []
    if not obs_id or not targets or not supp:
        continue
    m = OBS_MODELS.setdefault(obs_id, {"targets": set(), "supporting": set()})
    m["targets"].update(targets)
    m["supporting"].update(supp)

print("OBS_MODELS built for", len(OBS_MODELS), "observations with targeting.")


OBS_MODELS built for 37 observations with targeting.


In [5]:
def default_targeted_likelihood(obs_id: str, cause_id: str, value: str) -> float:
    # Generic likelihood if explicit likelihood is missing.
    # For targeted causes: supporting outcomes are likely (0.85), unknown small (0.05).
    # For non-target causes: supporting outcomes unlikely (0.15), unknown small (0.05).
    if obs_id not in obs_catalog:
        return 0.5
    vals = obs_catalog[obs_id]["values"]
    model = OBS_MODELS.get(obs_id)
    if not model:
        return 1.0 / len(vals)

    targets = model["targets"]
    supporting = model["supporting"]
    p_unknown = 0.05 if "unknown" in vals else 0.0
    supp_vals = [v for v in vals if v in supporting]
    other_vals = [v for v in vals if v not in supporting and v != "unknown"]

    if not supp_vals:
        return 1.0 / len(vals)

    if cause_id in targets:
        p_supp_total = 0.85
    else:
        p_supp_total = 0.15

    # distribute
    if value == "unknown":
        return p_unknown if p_unknown > 0 else 1.0/len(vals)
    if value in supp_vals:
        return p_supp_total / len(supp_vals)
    # remaining mass
    rem = max(0.0, 1.0 - p_unknown - p_supp_total)
    if other_vals:
        return rem / len(other_vals)
    return rem

def get_likelihood(obs_id: str, cause_id: str, value: str) -> float:
    # Prefer explicit likelihood tables if present
    vals = obs_catalog[obs_id]["values"]
    table = likelihoods.get(obs_id, {}).get(cause_id)
    if table is not None:
        return float(table.get(value, 1.0 / len(vals)))
    return default_targeted_likelihood(obs_id, cause_id, value)

def update_belief(belief: dict, obs_id: str, value: str) -> dict:
    upd = {}
    for c, p in belief.items():
        upd[c] = p * get_likelihood(obs_id, c, value)
    return normalize(upd)


In [6]:
def context_keywords(error_code: str, symptoms_yes: list[str], warnings: dict) -> set[str]:
    model = error_models[error_code]
    txt = (model.get("description","") + " " + " ".join(symptoms_yes)).lower()
    for oid, val in warnings.items():
        if val == "yes":
            txt += " " + oid.lower()
    keys = set()
    for k in UP_CFG.get("keywords", []):
        if k in txt:
            keys.add(k)
    return keys

def propose_upstream_causes(error_code: str, symptoms_yes: list[str], warnings: dict) -> list[str]:
    # Only add upstream if there is some context signal (symptoms or warnings)
    if not symptoms_yes and not warnings:
        return []
    listed = set(direct_cause_ids(error_code))
    keys = context_keywords(error_code, symptoms_yes, warnings)
    scored = []
    for cid, rc in root_causes.items():
        if cid in listed:
            continue
        label = (rc.get("label","") or "").lower()
        tags = set(rc.get("tags") or [])
        score = 0
        for k in keys:
            if k in label:
                score += 2
            if k.replace(" ", "_") in tags:
                score += 2
        if warnings.get("OBS_WARNING_290005_SEEN") == "yes" and "pump_tube" in tags:
            score += 3
        if warnings.get("OBS_WARNING_290020_SEEN") == "yes" and "ink_expired" in tags:
            score += 3
        if score > 0:
            scored.append((score, cid))
    scored.sort(reverse=True)
    max_n = UP_CFG.get("max_upstream_causes", 15)
    return [cid for _, cid in scored[:max_n]]

def build_initial_prior(error_code: str, symptoms_yes: list[str], warnings: dict):
    alpha = UP_CFG.get("alpha_error_code_mass", 0.90)
    direct = direct_cause_ids(error_code)
    model = error_models[error_code]

    # Start from direct priors (or uniform if missing)
    base = model.get("priors_root_cause_given_error", {}) or {cid: 1.0 for cid in direct}
    direct_prior = normalize({cid: float(base.get(cid, 0.0)) for cid in direct})

    belief = {cid: 0.0 for cid in root_causes.keys()}

    upstream = propose_upstream_causes(error_code, symptoms_yes, warnings)
    other_mass = max(0.0, 1.0 - alpha)

    if upstream:
        share = other_mass / len(upstream)
        for cid in upstream:
            belief[cid] += share
    else:
        # no upstream candidates -> spread remaining over nothing (keeps mass on direct)
        pass

    for cid, p in direct_prior.items():
        belief[cid] += alpha * p

    return normalize(belief), upstream


In [7]:
def apply_penalties(belief: dict, penalty: dict) -> dict:
    out = {}
    for cid, p in belief.items():
        out[cid] = p * penalty.get(cid, 1.0)
    return normalize(out)

def recompute_posterior(prior: dict, evidence_state: dict, penalty: dict) -> dict:
    b = prior.copy()
    for obs_id, value in evidence_state.items():
        if obs_id in obs_catalog:
            b = update_belief(b, obs_id, value)
    return apply_penalties(b, penalty)

def apply_near_zero_on_negative(proc_id: str, outcome: str, penalty: dict):
    proc = procedures[proc_id]
    targets = proc.get("targets_root_cause_ids") or []
    supp = proc.get("supporting_outcomes") or []
    if not targets or not supp:
        return
    if outcome not in supp:
        for cid in targets:
            penalty[cid] = penalty.get(cid, 1.0) * NEAR_ZERO


In [8]:
def is_warning_check(proc_id: str) -> bool:
    return bool(procedures[proc_id].get(WARNING_CHECK_FLAG))

def warning_check_allowed(proc_id: str, evidence_state: dict) -> bool:
    # Only allow if its produced warning obs is not already in evidence_state
    obs_id = procedures[proc_id].get("produces_observation")
    if obs_id in WARNING_OBS_IDS:
        return obs_id not in evidence_state
    return True

def confirm_done_for(repair_id: str, tried: set) -> bool:
    required = CONFIRM_MAP.get(repair_id)
    if required:
        return any(pid in tried for pid in required)
    if CONFIRM_FALLBACK == "any_check_or_test":
        return any(procedures[pid]["kind"] in ("check","test") for pid in tried)
    return False

def allowed_procedures(error_code: str, tried: set, evidence_state: dict) -> list[str]:
    cand = error_models[error_code].get("candidate_procedure_ids", [])
    out = []
    for pid in cand:
        if pid in tried or pid not in procedures:
            continue
        kind = procedures[pid].get("kind")
        if kind not in ("check","test","repair"):
            continue
        if is_warning_check(pid) and not warning_check_allowed(pid, evidence_state):
            continue
        # expensive repair gating
        if kind == "repair" and procedures[pid].get("is_expensive", False):
            if not confirm_done_for(pid, tried):
                continue
        out.append(pid)
    return out

def focus_direct_causes(belief: dict, error_code: str, k: int = 3) -> list[str]:
    direct = direct_cause_ids(error_code)
    ranked = sorted([(cid, belief.get(cid, 0.0)) for cid in direct], key=lambda x: x[1], reverse=True)
    return [cid for cid,_ in ranked[:k] if _>0]

def filter_focused_procedures(pids: list[str], focus_causes: list[str]) -> list[str]:
    # Keep checks/tests that target focused causes.
    focused = []
    for pid in pids:
        proc = procedures[pid]
        if proc["kind"] in ("check","test"):
            targets = set(proc.get("targets_root_cause_ids") or [])
            if targets and targets.intersection(focus_causes):
                focused.append(pid)
        else:
            focused.append(pid)  # repairs stay
    # If we found any focused checks/tests, drop non-focused checks/tests
    if any(procedures[pid]["kind"] in ("check","test") for pid in focused):
        return focused
    return pids


In [9]:
def predicted_outcome_distribution(belief: dict, proc_id: str) -> dict:
    proc = procedures[proc_id]
    vals = proc["outcomes"]
    dist = {v: 0.0 for v in vals}
    if proc["kind"] == "repair":
        # repairs are not scored by IG (we use repair-first)
        for v in vals:
            dist[v] = 1.0/len(vals)
        return dist
    obs_id = proc["produces_observation"]
    for v in vals:
        for c, p in belief.items():
            dist[v] += p * get_likelihood(obs_id, c, v)
    return normalize(dist)

def expected_entropy_after(belief: dict, proc_id: str) -> float:
    proc = procedures[proc_id]
    obs_id = proc["produces_observation"]
    out_dist = predicted_outcome_distribution(belief, proc_id)
    expH = 0.0
    for v, pv in out_dist.items():
        if pv <= 1e-12:
            continue
        post = update_belief(belief, obs_id, v)
        expH += pv * entropy(post)
    return expH

def score_ig(belief: dict, proc_id: str, lambda_time: float, lambda_parts: float) -> dict:
    proc = procedures[proc_id]
    H0 = entropy(belief)
    expH = expected_entropy_after(belief, proc_id)
    ig = H0 - expH
    cost = proc["cost"]
    score = ig - lambda_time * cost["time_min"] - lambda_parts * cost["parts_eur"]
    return {"procedure_id": proc_id, "mode":"ig", "score": score, "ig": ig, "time_min": cost["time_min"], "parts_eur": cost["parts_eur"]}

def estimated_resolve_prob(belief: dict, repair_id: str) -> float:
    # If action_success exists, use it; otherwise use target overlap heuristic.
    if action_success:
        p = 0.0
        for c, bc in belief.items():
            table = action_success.get(repair_id, {}).get(c)
            if table:
                p += bc * float(table.get("error_resolved", table.get("yes", 0.5)))
            else:
                p += bc * 0.45
        return p
    targets = set(procedures[repair_id].get("targets_root_cause_ids") or [])
    p = 0.0
    for c, bc in belief.items():
        p += bc * (0.85 if c in targets else 0.15)
    return p

def score_repair_utility(belief: dict, repair_id: str, lambda_time: float, lambda_parts: float) -> dict:
    proc = procedures[repair_id]
    p_res = estimated_resolve_prob(belief, repair_id)
    cost = proc["cost"]
    denom = 1.0 + lambda_time * cost["time_min"] + lambda_parts * cost["parts_eur"]
    score = p_res / denom
    return {"procedure_id": repair_id, "mode":"repair_first", "score": score, "p_resolve": p_res, "time_min": cost["time_min"], "parts_eur": cost["parts_eur"]}


In [10]:
def find_part_by_keyword(keyword: str):
    k = keyword.lower()
    for part_id, p in parts.items():
        desc = (p.get("description") or p.get("Description") or "").lower()
        if k in desc:
            return part_id
    return None

# Heuristic part mapping for padding to top-5 if KB previsit list is short
KEY_PARTS = {
    "pump_tube": find_part_by_keyword("pump tube"),
    "printhead": find_part_by_keyword("printhead"),
    "air_filter": find_part_by_keyword("filter,air") or find_part_by_keyword("filter"),
    "air_tube": find_part_by_keyword("tube,air") or find_part_by_keyword("tube,air"),
    "ink_tube_placeholder": "PART_Z_0502_0001",  # placeholder (tube list not provided)
}

def show_previsit_parts(error_code: str, initial_evidence: dict | None = None):
    initial_evidence = dict(initial_evidence or {})
    symptoms_yes = initial_evidence.pop("__symptoms__", [])
    warnings = {k:v for k,v in initial_evidence.items() if k in WARNING_OBS_IDS}

    prior, _ = build_initial_prior(error_code, symptoms_yes, warnings)
    belief = recompute_posterior(prior, initial_evidence, penalty={})

    topn = top_k(belief, ALT_TOPN)
    high = [(cid,p) for cid,p in topn if p >= ALT_TH]
    show_alt = len(high) >= ALT_MIN

    base = list(error_models[error_code].get("previsit_part_ids", []))

    # Pad using direct causes -> likely parts
    direct = direct_cause_ids(error_code)
    direct_labels = " ".join(root_causes[cid]["label"].lower() for cid in direct if cid in root_causes)

    pad=[]
    if "pump tube" in direct_labels and KEY_PARTS["pump_tube"]:
        pad.append(KEY_PARTS["pump_tube"])
    if "printhead" in direct_labels and KEY_PARTS["printhead"]:
        pad.append(KEY_PARTS["printhead"])
    if "air filter" in direct_labels and KEY_PARTS["air_filter"]:
        pad.append(KEY_PARTS["air_filter"])
    if "air" in direct_labels and KEY_PARTS["air_tube"]:
        pad.append(KEY_PARTS["air_tube"])
    if "ink tube" in direct_labels:
        pad.append(KEY_PARTS["ink_tube_placeholder"])

    # Merge + unique preserving order
    merged=[]
    for pid in base + pad:
        if pid and pid not in merged:
            merged.append(pid)

    merged = merged[:PV_TOPK]

    print(f"Pre-visit parts for {error_code} (top {PV_TOPK})")
    print(f"Alternatives shown? {show_alt}  (>= {ALT_MIN} of top-{ALT_TOPN} causes have p>= {ALT_TH})")
    if show_alt:
        for cid,p in high:
            print(f"  - {p:.3f} {cid} | {root_causes[cid]['label']}")

    for part_id in merged:
        pinfo = parts.get(part_id, {"part_id": part_id, "description": "(placeholder / not in CSV)"})
        desc = pinfo.get("description") or pinfo.get("Description") or "(no description)"
        pn = pinfo.get("part_number") or pinfo.get("PartNumber") or ""
        print(f"- {part_id} | {pn} | {desc}")

    return merged


In [11]:
def run_session(
    error_code: str,
    symptoms_yes=None,
    initial_warnings=None,
    initial_counters=None,
    max_steps: int = 20,
    lambda_time: float = 0.02,
    lambda_parts: float = 0.005,
    log_dir: str = "logs"
):
    symptoms_yes = symptoms_yes or []
    initial_warnings = initial_warnings or {}
    initial_counters = initial_counters or {}

    session_id = str(uuid.uuid4())
    ts = now_utc_iso()

    tried = set()
    evidence_state = {}
    penalty = {}  # cause_id -> multiplier (near zero)
    history = []

    prior, upstream = build_initial_prior(error_code, symptoms_yes, initial_warnings)

    # Evidence as state (overwrite semantics)
    for s in symptoms_yes:
        oid = "OBS_SYM_" + re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_").upper()[:50]
        if oid not in obs_catalog:
            obs_catalog[oid] = {"observation_id": oid, "label": f"Symptom: {s}", "type":"binary", "values":["yes","no"]}
        evidence_state[oid] = "yes"

    for oid, val in initial_warnings.items():
        evidence_state[oid] = val

    for oid, val in initial_counters.items():
        evidence_state[oid] = val

    belief = recompute_posterior(prior, evidence_state, penalty)

    print("\nSESSION", session_id, "| error_code", error_code)
    print("epsilon:", EPSILON, "| recommend:", REPAIR_RECOMMEND, "| consider:", REPAIR_CONSIDER)
    print("lambda_time:", lambda_time, "| lambda_parts:", lambda_parts)
    if upstream:
        print("Upstream causes added:", len(upstream))

    for step in range(1, max_steps + 1):
        print("\n--- Step", step, "---")
        top = top_k(belief, 8)
        for cid, p in top:
            tag = " (UPSTREAM)" if is_upstream(cid, error_code) else ""
            print(f"{p:.3f}  {cid}{tag} | {root_causes[cid]['label']}")

        if all(p < EPSILON for p in belief.values()):
            print("\nAll hypotheses < epsilon. ESCALATE.")
            history.append({"type":"stop","reason":"exhausted"})
            break

        focus = focus_direct_causes(belief, error_code, k=3)
        allowed = allowed_procedures(error_code, tried, evidence_state)
        allowed = filter_focused_procedures(allowed, focus)

        if not allowed:
            print("\nNo more allowed procedures. ESCALATE.")
            history.append({"type":"stop","reason":"no_procedures"})
            break

        top_c, top_p = top[0]

        chosen = None

        # Repair-first if confident: choose best utility among repairs that target the top cause (if any)
        if top_p >= REPAIR_RECOMMEND:
            repairs = [pid for pid in allowed if procedures[pid]["kind"] == "repair" and procedures[pid].get("repair_role","resolution")=="resolution"]
            if repairs:
                # prefer those targeting top cause
                targeting = [r for r in repairs if top_c in set(procedures[r].get("targets_root_cause_ids") or [])]
                cand_repairs = targeting or repairs
                scored = [score_repair_utility(belief, r, lambda_time, lambda_parts) for r in cand_repairs]
                scored.sort(key=lambda d: d["score"], reverse=True)
                chosen = scored[0]

        if chosen is None:
            # Score checks/tests by IG; include precheck repairs only if nothing else left
            scored = []
            for pid in allowed:
                proc = procedures[pid]
                if proc["kind"] in ("check","test"):
                    scored.append(score_ig(belief, pid, lambda_time, lambda_parts))
            if scored:
                scored.sort(key=lambda d: d["score"], reverse=True)
                chosen = scored[0]
            else:
                # fallback: if only repairs remain
                repairs = [pid for pid in allowed if procedures[pid]["kind"]=="repair"]
                scored = [score_repair_utility(belief, r, lambda_time, lambda_parts) for r in repairs]
                scored.sort(key=lambda d: d["score"], reverse=True)
                chosen = scored[0]

        pid = chosen["procedure_id"]
        proc = procedures[pid]

        snap = {
            "top_cause_id": top_c,
            "top_cause_p": top_p,
            "selected_procedure_id": pid,
            "selected_procedure_label": proc["label"],
            "selection": chosen
        }

        print(f"\nNEXT: {pid} | {proc['label']}")
        if chosen["mode"] == "repair_first":
            print(f"  mode=repair_first score={chosen['score']:.4f} p_resolve={chosen['p_resolve']:.3f} time={chosen['time_min']}m parts=€{chosen['parts_eur']}")
        else:
            print(f"  mode=ig score={chosen['score']:.4f} IG={chosen['ig']:.4f} time={chosen['time_min']}m parts=€{chosen['parts_eur']}")
        print("  outcomes:", proc["outcomes"])

        outcome = input("Enter outcome value: ").strip()
        if outcome not in proc["outcomes"]:
            print("Invalid outcome; try again.")
            continue

        tried.add(pid)

        # Precheck actions
        if proc["kind"] == "repair" and proc.get("repair_role") == "precheck":
            if outcome == "not_done":
                # skip current focus/top cause (near-zero)
                penalty[top_c] = penalty.get(top_c, 1.0) * NEAR_ZERO
            history.append({"type":"precheck_action","procedure_id":pid,"outcome":outcome,"snapshot":snap})
            belief = recompute_posterior(prior, evidence_state, penalty)
            continue

        if proc["kind"] == "repair":
            # record resolved/not resolved as observation
            evidence_state["OBS_RESOLVED"] = outcome
            history.append({"type":"repair","procedure_id":pid,"outcome":outcome,"snapshot":snap})
            if outcome == "error_resolved":
                print("\nResolved. STOP.")
                history.append({"type":"stop","reason":"resolved"})
                break
            else:
                # error not resolved is evidence; downweight top cause slightly and continue
                penalty[top_c] = penalty.get(top_c, 1.0) * 0.3
                belief = recompute_posterior(prior, evidence_state, penalty)
                continue

        # Check/test: apply near-zero penalties for negative targeted evidence
        apply_near_zero_on_negative(pid, outcome, penalty)

        obs_id = proc["produces_observation"]
        evidence_state[obs_id] = outcome
        history.append({"type":proc["kind"],"procedure_id":pid,"observation_id":obs_id,"value":outcome,"snapshot":snap})

        belief = recompute_posterior(prior, evidence_state, penalty)

    # Log
    log_entry = {
        "session_id": session_id,
        "timestamp_utc": ts,
        "error_code": error_code,
        "symptoms_yes": symptoms_yes,
        "initial_warnings": initial_warnings,
        "initial_counters": initial_counters,
        "dynamic_upstream_causes": upstream,
        "evidence_state_final": evidence_state,
        "penalty_final": penalty,
        "history": history,
        "final_top_causes": [{"cause_id": cid, "p": p, "label": root_causes[cid]["label"]} for cid, p in top_k(belief, 10)],
        "epsilon": EPSILON,
        "lambda_time": lambda_time,
        "lambda_parts": lambda_parts
    }

    Path(log_dir).mkdir(parents=True, exist_ok=True)
    log_path = Path(log_dir) / "diagnostic_sessions.jsonl"
    with log_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(log_entry, ensure_ascii=False) + "\n")

    print("\nLog saved to:", log_path.resolve())
    return log_entry


## Example

Run the cell below.

Expected improvements vs your last run:
- Direct causes will NOT be marked as UPSTREAM anymore.
- Warnings will NOT be re-asked if you provide them.
- Early checks will focus on the top-ranked **direct** causes.
- Probabilities will start moving because we use targeted default likelihoods + near-zero penalties.


In [12]:
show_previsit_parts("250001", {
  "__symptoms__": ["Ink delivery to printhead failed"],
  "OBS_WARNING_290005_SEEN": "yes",
  "OBS_WARNING_290020_SEEN": "no",
  "OBS_LIFE_PUMP_TUBE": "high",
  "OBS_LIFE_INK_PUMP": "medium"
})

run_session(
  "250001",
  symptoms_yes=["Ink delivery to printhead failed"],
  initial_warnings={"OBS_WARNING_290005_SEEN":"yes","OBS_WARNING_290020_SEEN":"no"},
  initial_counters={"OBS_LIFE_PUMP_TUBE":"high","OBS_LIFE_INK_PUMP":"medium"}
)


Pre-visit parts for 250001 (top 5)
Alternatives shown? False  (>= 2 of top-5 causes have p>= 0.2)
- PART_Z_0203_0064 |  | Pump tube
- PART_Z_0402_0001 | 1070129581 | UVGEL 425 PRINTHEAD (1 PCS)
- PART_Z_0502_0001 |  | Ink tube (range)
- PART_Z_0202_0045 | 1060032111 | FILTER,AIR
- PART_Z_0402_0003 | 1070077219 | TUBE,AIR (PACKED)

SESSION 0bcedf13-02cf-4605-b11f-67fb1e4880d1 | error_code 250001
epsilon: 0.02 | recommend: 0.8 | consider: 0.75
lambda_time: 0.02 | lambda_parts: 0.005
Upstream causes added: 15

--- Step 1 ---
0.430  RC_WORN_OUT_PUMP_TUBE | Worn out pump tube
0.076  RC_BLOCKED_PRINTHEAD_VALVE | Blocked printhead valve
0.076  RC_INK_LEAKAGE | Ink Leakage
0.076  RC_FAULTY_PRINTHEAD | Faulty printhead
0.076  RC_CLOGGED_INK_TUBE | Clogged ink tube
0.076  RC_CLOGGED_AIR_FILTER | Clogged air filter
0.076  RC_FAULTY_BOILER_UNIT | Faulty boiler unit
0.022  RC_WRONG_POSITION_OF_PUMP_TUBE (UPSTREAM) | Wrong position of pump tube.

NEXT: PRC_CHECK_CHECK_PUMP_TUBE | Check pump tube
  m

{'session_id': '0bcedf13-02cf-4605-b11f-67fb1e4880d1',
 'timestamp_utc': '2026-03-10T14:26:44.100291Z',
 'error_code': '250001',
 'symptoms_yes': ['Ink delivery to printhead failed'],
 'initial_warnings': {'OBS_WARNING_290005_SEEN': 'yes',
  'OBS_WARNING_290020_SEEN': 'no'},
 'initial_counters': {'OBS_LIFE_PUMP_TUBE': 'high',
  'OBS_LIFE_INK_PUMP': 'medium'},
 'dynamic_upstream_causes': ['RC_BLOCKAGE_IN_INK_TUBE_FROM_3_WAY_VALVE_TO_PRINTHEAD',
  'RC_PRESSURE_FAILURE_INSIDE_THE_PRINTHEAD_DUE_TO_A_BLOCKED_AIR_TUBE',
  'RC_CLOSED_INK_VALVES_AT_PRINTHEAD',
  'RC_FAULT_INK_LEAK_SENSOR',
  'RC_FAULTY_ELECTRICAL_CONNECTION_OF_INK_LEAK_SENSOR',
  'RC_WRONG_POSITION_OF_PUMP_TUBE',
  'RC_PUMP_TUBE_RECENTLY_REPLACED_WITHOUT_RESETTING_THE_MAINTENANCE_COUNTER',
  'RC_FAULTY_PUMP_TUBE',
  'RC_OVER_PRESSURE_VALVE_DAMAGED',
  'RC_INK_LOW_SENSOR_DEFECT',
  'RC_INK_HIGH_SENSOR_DEFECT',
  'RC_FAULLTY_INK_LEVEL_SENSOR',
  'RC_BLOCKAGE_IN_THE_PRINT_HEAD_BOILER_TUBE',
  'RC_BLOCKADE_IN_WHITE_INK_RECIRCULATI